# Feature Engineering for UPI Fraud Detection

This notebook transforms synthetic UPI transaction logs into behavioral features for unsupervised anomaly detection.

The goal is to help an Isolation Forest model learn each user's normal transaction fingerprint and flag transactions that deviate from normal behavior.

## 1. Imports and Project Paths

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SYNTHETIC_DATA_PATH = PROJECT_ROOT / "data" / "synthetic" / "upi_transactions.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    "hour_of_day",
    "day_of_week",
    "transaction_value",
    "is_first_time_receiver",
    "velocity_last_10min",
    "amount_vs_user_avg",
]

TARGET_COLUMN = "is_fraud"

## 2. Load Simulated UPI Transactions

In [2]:
def load_transactions(file_path: Path) -> pd.DataFrame:
    """Load synthetic UPI transactions and parse timestamps."""
    if not file_path.exists():
        raise FileNotFoundError(
            f"Transaction file not found: {file_path}. Run the data simulation notebook first."
        )

    transactions = pd.read_csv(file_path, parse_dates=["timestamp"])
    return transactions.sort_values("timestamp").reset_index(drop=True)


transactions = load_transactions(SYNTHETIC_DATA_PATH)

print(f"Loaded transactions: {transactions.shape}")
transactions.head()

Loaded transactions: (10000, 8)


,transaction_id,timestamp,sender_account_id,receiver_account_id,amount,location_pincode,transaction_type,is_fraud
0,TXN_005083,2024-01-01 00:00:00,ACC_1001,RCV_4436,1487.51,560001,Merchant,0
1,TXN_008283,2024-01-01 00:50:00,ACC_4572,RCV_7208,2711.60,560001,P2P,0
2,TXN_002316,2024-01-01 01:05:00,ACC_4271,RCV_5491,497.85,560001,P2P,0
3,TXN_003674,2024-01-01 01:10:00,ACC_4711,RCV_3991,3050.08,560001,P2P,0
4,TXN_007006,2024-01-01 01:20:00,ACC_4038,RCV_4270,5436.65,700001,P2P,0


## 3. Validate Input Schema

In [3]:
def validate_input_schema(transactions: pd.DataFrame) -> None:
    """Ensure all columns required for feature engineering are present."""
    required_columns = {
        "transaction_id",
        "timestamp",
        "sender_account_id",
        "receiver_account_id",
        "amount",
        "location_pincode",
        "transaction_type",
        TARGET_COLUMN,
    }
    missing_columns = required_columns.difference(transactions.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    if transactions["transaction_id"].duplicated().any():
        raise ValueError("Transaction IDs must be unique before feature engineering.")
    if transactions["timestamp"].isna().any():
        raise ValueError("Timestamp values must not be missing.")
    if (transactions["amount"] <= 0).any():
        raise ValueError("Transaction amounts must be positive.")


validate_input_schema(transactions)
print("Input schema validation passed.")

Input schema validation passed.


## 4. Engineer Suspicion Features

In [4]:
def add_time_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Extract time-based signals from the transaction timestamp."""
    features = transactions.copy()
    features["hour_of_day"] = features["timestamp"].dt.hour
    features["day_of_week"] = features["timestamp"].dt.dayofweek
    return features


def add_first_time_receiver_feature(transactions: pd.DataFrame) -> pd.DataFrame:
    """Flag whether a sender is paying a receiver for the first time."""
    features = transactions.sort_values(["sender_account_id", "timestamp"]).copy()
    previous_pair_seen = features.duplicated(
        subset=["sender_account_id", "receiver_account_id"],
        keep="first",
    )
    features["is_first_time_receiver"] = (~previous_pair_seen).astype(int)
    return features.sort_index()


def add_amount_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Create transaction amount and sender-average amount deviation features."""
    features = transactions.copy()
    sender_average_amount = features.groupby("sender_account_id")["amount"].transform("mean")

    features["transaction_value"] = features["amount"]
    features["amount_vs_user_avg"] = features["amount"] / sender_average_amount.clip(lower=1)
    return features


def calculate_sender_velocity(transactions: pd.DataFrame) -> pd.Series:
    """Count each sender's transactions in the previous 10 minutes."""
    sorted_transactions = transactions.sort_values(["sender_account_id", "timestamp"])
    velocity = pd.Series(index=sorted_transactions.index, dtype=float)

    for _, group in sorted_transactions.groupby("sender_account_id"):
        rolling_counts = (
            group.set_index("timestamp")["transaction_id"]
            .rolling("10min")
            .count()
            .to_numpy()
        )
        velocity.loc[group.index] = rolling_counts

    return velocity.reindex(transactions.index).fillna(1)


def add_velocity_feature(transactions: pd.DataFrame) -> pd.DataFrame:
    """Add transaction velocity as a fraud-risk behavior signal."""
    features = transactions.copy()
    features["velocity_last_10min"] = calculate_sender_velocity(features)
    return features


def build_behavioral_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Build all model-ready behavioral features."""
    features = transactions.copy()
    features = add_time_features(features)
    features = add_first_time_receiver_feature(features)
    features = add_amount_features(features)
    features = add_velocity_feature(features)
    return features


feature_dataset = build_behavioral_features(transactions)
feature_dataset[FEATURE_COLUMNS + [TARGET_COLUMN]].head()

,hour_of_day,day_of_week,transaction_value,is_first_time_receiver,velocity_last_10min,amount_vs_user_avg,is_fraud
0,0,0,1487.51,1,1.0,1.000000,0
1,0,0,2711.60,1,1.0,1.219036,0
2,1,0,497.85,1,1.0,0.337527,0
3,1,0,3050.08,1,1.0,1.551246,0
4,1,0,5436.65,1,1.0,1.000000,0


## 5. Prepare Model Matrix

In [5]:
def prepare_model_data(feature_dataset: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Split engineered data into model features and evaluation labels."""
    missing_features = set(FEATURE_COLUMNS).difference(feature_dataset.columns)
    if missing_features:
        raise ValueError(f"Missing engineered features: {sorted(missing_features)}")

    features = feature_dataset[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0)
    labels = feature_dataset[TARGET_COLUMN].astype(int)
    return features, labels


X, y = prepare_model_data(feature_dataset)

print(f"Feature matrix shape: {X.shape}")
print(f"Label vector shape: {y.shape}")
X.describe().round(2)

Feature matrix shape: (10000, 6)
Label vector shape: (10000,)


,hour_of_day,day_of_week,transaction_value,is_first_time_receiver,velocity_last_10min,amount_vs_user_avg
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,11.18,2.99,5664.11,1.00,1.00,1.00
std,6.97,1.99,22943.16,0.01,0.02,0.64
min,0.00,0.00,17.40,0.00,1.00,0.00
25%,5.00,1.00,954.61,1.00,1.00,0.60
50%,11.00,3.00,1650.54,1.00,1.00,1.00
75%,17.00,5.00,2648.75,1.00,1.00,1.28
max,23.00,6.00,198754.15,1.00,2.00,8.59


## 6. Feature Quality Checks

In [6]:
def summarize_feature_quality(features: pd.DataFrame, labels: pd.Series) -> pd.DataFrame:
    """Summarize nulls, unique values, and label-wise means for each feature."""
    normal_means = features.loc[labels == 0].mean()
    fraud_means = features.loc[labels == 1].mean()

    return pd.DataFrame(
        {
            "missing_values": features.isna().sum(),
            "unique_values": features.nunique(),
            "normal_mean": normal_means,
            "fraud_mean": fraud_means,
            "fraud_to_normal_mean_ratio": fraud_means / normal_means.replace(0, np.nan),
        }
    ).round(3)


feature_quality_summary = summarize_feature_quality(X, y)
feature_quality_summary

,missing_values,unique_values,normal_mean,fraud_mean,fraud_to_normal_mean_ratio
hour_of_day,0,24,11.448,2.460,0.215
day_of_week,0,7,2.986,3.107,1.040
transaction_value,0,9871,1885.539,127837.879,67.799
is_first_time_receiver,0,2,1.000,1.000,1.000
velocity_last_10min,0,2,1.000,1.000,1.000
amount_vs_user_avg,0,8661,0.942,2.877,3.054


In [7]:
feature_dataset.groupby(TARGET_COLUMN)[FEATURE_COLUMNS].mean().round(2)

,hour_of_day,day_of_week,transaction_value,is_first_time_receiver,velocity_last_10min,amount_vs_user_avg
is_fraud,,,,,,
0,11.45,2.99,1885.54,1.0,1.0,0.94
1,2.46,3.11,127837.88,1.0,1.0,2.88


## 7. Save Processed Datasets

In [8]:
def save_processed_outputs(
    feature_dataset: pd.DataFrame,
    features: pd.DataFrame,
    labels: pd.Series,
    output_dir: Path,
) -> dict[str, Path]:
    """Persist engineered datasets for model training and evaluation."""
    output_paths = {
        "feature_dataset": output_dir / "upi_transactions_with_features.csv",
        "feature_matrix": output_dir / "feature_matrix.csv",
        "labels": output_dir / "labels.csv",
    }

    feature_dataset.to_csv(output_paths["feature_dataset"], index=False)
    features.to_csv(output_paths["feature_matrix"], index=False)
    labels.to_frame(name=TARGET_COLUMN).to_csv(output_paths["labels"], index=False)
    return output_paths


saved_paths = save_processed_outputs(feature_dataset, X, y, PROCESSED_DATA_DIR)

for name, path in saved_paths.items():
    print(f"Saved {name}: {path}")

Saved feature_dataset: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\data\processed\upi_transactions_with_features.csv
Saved feature_matrix: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\data\processed\feature_matrix.csv
Saved labels: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\data\processed\labels.csv


## 8. Output Summary

Generated files:

- `data/processed/upi_transactions_with_features.csv`
- `data/processed/feature_matrix.csv`
- `data/processed/labels.csv`

These files are ready for Isolation Forest training and F1-score threshold tuning.